# Implement Gradient Checkpointing — Solution

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google, NVIDIA, Tesla

---

### Problem Statement

Gradient checkpointing (also called activation checkpointing or rematerialization) is a critical memory optimization technique for training deep neural networks. During the forward pass, instead of storing all intermediate activations needed for backpropagation, we discard them and **recompute** them during the backward pass.

This trades compute for memory. The price is roughly one extra forward pass per training step (about 20–33% more compute, since backward costs about twice the forward). The saving depends on *what* you checkpoint: checkpointing every block, as here, keeps only each block's input (still O(N) tensors for N blocks, but none of the intermediates inside a block), plus one block's intermediates at a time while backward recomputes it. Checkpointing segments of √N layers gives the classic O(√N) activation memory (Chen et al., 2016). This technique is essential for training large language models and other deep architectures that would otherwise not fit in GPU memory.

Your task is to implement gradient checkpointing using a custom `torch.autograd.Function` that saves only inputs during forward and recomputes activations during backward.

---

### Requirements

1. **CheckpointFunction** — A custom `torch.autograd.Function` whose forward saves only inputs (not intermediate activations), and whose backward recomputes the forward pass to recover gradients.
2. **checkpoint** — A wrapper function that applies gradient checkpointing to any callable.
3. **Deep Network** — Build a 20+ layer network, compare memory usage with and without checkpointing.
4. **Correctness** — Gradients with checkpointing must be identical to gradients without, including for the parameters that `fn` closes over and when the input does not require grad.
5. **Randomness** — If `fn` uses random ops (dropout), the recomputation in backward must draw the same random numbers as the original forward. Save the RNG state in forward and restore it for the recomputation, as `torch.utils.checkpoint` does with `preserve_rng_state=True`.

---

### Constraints

- ✅ Must work with any function of tensor arguments (not just nn.Module), including ones that use dropout.
- ✅ Must correctly handle the autograd graph (gradients flow through the checkpoint).
- ✅ Gradients must match the non-checkpointed run. Recomputation replays the same ops, so with deterministic kernels they are bitwise identical; the tests allow float tolerance.
- ❌ Do **not** use `torch.utils.checkpoint` in your implementation.

---

<details>
  <summary>💡 Hint</summary>

  - In `CheckpointFunction.forward`, use `ctx.save_for_backward(*inputs)` to save only the inputs. Autograd is already disabled inside `autograd.Function.forward`, so `fn(*args)` records no graph and keeps no intermediates; wrapping it in `torch.no_grad()` makes that explicit.
  - In `CheckpointFunction.backward`, re-enable gradients with `torch.enable_grad()`, rerun the function on detached copies of the saved inputs, then call `torch.autograd.backward(outputs, grad_outputs)`. Unlike `torch.autograd.grad()` on the inputs alone, this also fills in `.grad` for the parameters `fn` closes over; read the input gradients back off the detached copies.
  - Autograd only calls `backward` if some tensor *argument* requires grad. The first block is usually fed plain data, so in `checkpoint` pass an extra dummy tensor with `requires_grad=True` when no argument requires grad, or that block's weights never train.
  - Save the function itself on the context: `ctx.fn = fn`.
  - Dropout: stash `torch.get_rng_state()` (and the CUDA RNG state if CUDA is in use) in forward. In backward, restore it inside `torch.random.fork_rng()` before recomputing, so the replay draws the same mask without disturbing the RNG stream of the rest of training.
  - Memory: `torch.autograd.graph.saved_tensors_hooks` sees every tensor autograd saves for backward, which is exactly what checkpointing removes. On a GPU, `torch.cuda.max_memory_allocated()` measures the peak.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Deep network for testing
class DeepBlock(nn.Module):
    """A single residual block: Linear -> ReLU -> Linear + residual."""
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, x):
        residual = x
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x + residual


class DeepNetwork(nn.Module):
    """A deep network with many residual blocks."""
    def __init__(self, dim=256, num_blocks=24, use_checkpoint=False):
        super().__init__()
        self.blocks = nn.ModuleList([DeepBlock(dim) for _ in range(num_blocks)])
        self.head = nn.Linear(dim, 1)
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        for block in self.blocks:
            if self.use_checkpoint:
                x = checkpoint(block, x)
            else:
                x = block(x)
        return self.head(x)


torch.manual_seed(42)
print("Deep network blocks: 24")
print(f"Parameters per block: {sum(p.numel() for p in DeepBlock(256).parameters()):,}")

In [ ]:
class CheckpointFunction(torch.autograd.Function):
    """
    Custom autograd function that implements gradient checkpointing.

    Forward: Run the function, saving only its inputs (and the RNG state).
    Backward: Recompute the forward pass to get intermediates, then compute gradients.
    """

    @staticmethod
    def forward(ctx, fn, *args):
        """
        Run fn(*args) but only save the inputs for backward.
        """
        ctx.fn = fn
        # Save only the inputs; intermediates are recomputed in backward
        ctx.save_for_backward(*args)
        # Remember the RNG state so random ops inside fn (dropout) draw the same
        # numbers when fn is recomputed in backward
        ctx.cpu_rng_state = torch.get_rng_state()
        ctx.cuda_rng_state = torch.cuda.get_rng_state() if torch.cuda.is_initialized() else None

        # Autograd is already off inside Function.forward; no_grad makes it explicit:
        # no graph is recorded, so no intermediate activation is kept alive.
        with torch.no_grad():
            return fn(*args)

    @staticmethod
    def backward(ctx, *grad_outputs):
        """
        Recompute forward pass and compute gradients.
        """
        # Fresh leaves to recompute from (only those that required grad need grads)
        detached_inputs = [inp.detach().requires_grad_(inp.requires_grad)
                           for inp in ctx.saved_tensors]

        # Replay the forward's RNG state; fork_rng restores the current state on exit,
        # so the rest of training sees an undisturbed random stream.
        devices = [torch.cuda.current_device()] if ctx.cuda_rng_state is not None else []
        with torch.random.fork_rng(devices=devices):
            torch.set_rng_state(ctx.cpu_rng_state)
            if ctx.cuda_rng_state is not None:
                torch.cuda.set_rng_state(ctx.cuda_rng_state)
            with torch.enable_grad():
                output = ctx.fn(*detached_inputs)

        # Use autograd.backward, NOT autograd.grad on the inputs alone.
        # fn normally closes over module parameters, and those parameters are not
        # in `inputs`. autograd.grad would return gradients only for the tensor
        # arguments, so every checkpointed layer's weights would silently receive
        # no gradient and never train. autograd.backward accumulates into every
        # leaf reached during recomputation - parameters included - and we read
        # the input gradients back off the detached copies.
        outputs = output if isinstance(output, tuple) else (output,)
        # Skip outputs that do not depend on anything trainable (e.g. a frozen fn).
        pairs = [(o, g) for o, g in zip(outputs, grad_outputs)
                 if isinstance(o, torch.Tensor) and o.requires_grad]
        if pairs:
            torch.autograd.backward([o for o, _ in pairs], [g for _, g in pairs])

        # None for fn, then the gradient (or None) for each input
        return (None, *(inp.grad if inp.requires_grad else None for inp in detached_inputs))


def checkpoint(fn, *args):
    """
    Apply gradient checkpointing to fn(*args). args must be tensors.
    """
    # Autograd only records CheckpointFunction (and later calls its backward) if at
    # least one tensor *argument* requires grad. fn's parameters are not arguments,
    # so when the input is plain data (the first block, fed a batch straight from
    # the dataset) backward would never run and fn's weights would never train.
    # Passing a dummy tensor that requires grad keeps the checkpoint in the graph.
    if torch.is_grad_enabled() and not any(
        isinstance(a, torch.Tensor) and a.requires_grad for a in args
    ):
        dummy = torch.empty(0, requires_grad=True)
        return CheckpointFunction.apply(lambda _dummy, *a: fn(*a), dummy, *args)
    return CheckpointFunction.apply(fn, *args)


def estimate_memory(model, x):
    """
    Bytes of activations autograd saves for backward during one forward pass.

    saved_tensors_hooks sees every tensor any op (or ctx.save_for_backward) stores
    for backward. Parameters are excluded: they live in memory either way.
    """
    param_storages = {p.untyped_storage().data_ptr() for p in model.parameters()}
    saved = {}

    def pack(t):
        storage = t.untyped_storage()
        if storage.data_ptr() not in param_storages:
            saved[storage.data_ptr()] = storage.nbytes()   # count each buffer once
        return t

    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        model(x)
    return sum(saved.values())


In [ ]:
# === Validation ===
torch.manual_seed(42)

# 1. Gradient correctness test
print("=== Gradient Correctness Test ===")
dim = 256
x = torch.randn(16, dim, requires_grad=True)

# Without checkpointing
model_no_ckpt = DeepNetwork(dim=dim, num_blocks=24, use_checkpoint=False)
torch.manual_seed(42)
model_ckpt = DeepNetwork(dim=dim, num_blocks=24, use_checkpoint=True)
# Copy weights
model_ckpt.load_state_dict(model_no_ckpt.state_dict())

# Forward + backward without checkpointing
x1 = x.detach().clone().requires_grad_(True)
out1 = model_no_ckpt(x1)
loss1 = out1.sum()
loss1.backward()

# Forward + backward with checkpointing
x2 = x.detach().clone().requires_grad_(True)
out2 = model_ckpt(x2)
loss2 = out2.sum()
loss2.backward()

# Compare gradients
print(f"Output match: {torch.allclose(out1, out2, atol=1e-6)}")
print(f"Input grad match: {torch.allclose(x1.grad, x2.grad, atol=1e-5)}")

grad_match = True
for (n1, p1), (n2, p2) in zip(model_no_ckpt.named_parameters(), model_ckpt.named_parameters()):
    # A missing gradient is a FAILURE, not something to skip: checkpointed
    # parameters silently receiving no gradient is the classic bug here.
    if p1.grad is None or p2.grad is None:
        print(f"  MISSING gradient at {n1}: no_ckpt={p1.grad is not None}, ckpt={p2.grad is not None}")
        grad_match = False
    else:
        if not torch.allclose(p1.grad, p2.grad, atol=1e-5):
            print(f"  Gradient mismatch at {n1}: max diff = {(p1.grad - p2.grad).abs().max():.2e}")
            grad_match = False

assert grad_match, "Gradients should match between checkpointed and non-checkpointed!"
print("All parameter gradients match!")
print("PASSED\n")

# 2. Memory test: bytes autograd keeps alive for backward
print("=== Memory Test ===")
x_test = torch.randn(16, dim)
mem_no_ckpt = estimate_memory(model_no_ckpt, x_test)
mem_ckpt = estimate_memory(model_ckpt, x_test)
print(f"Saved for backward without checkpointing: {mem_no_ckpt / 1024:,.0f} KB")
print(f"Saved for backward with checkpointing:    {mem_ckpt / 1024:,.0f} KB")
print("Each block now keeps only its input instead of its input and the ReLU output;")
print("the more ops inside a checkpointed segment, the bigger the saving.")
assert mem_ckpt < mem_no_ckpt, "Checkpointing should reduce the activations saved for backward!"
print("PASSED\n")

# 3. Randomness: dropout inside a checkpointed function
print("=== Dropout Replay Test ===")
def dropout_block(t, w):
    return F.dropout(torch.tanh(t @ w), p=0.5, training=True)

w_plain = torch.randn(64, 64, requires_grad=True)
w_ckpt = w_plain.detach().clone().requires_grad_(True)
t_in = torch.randn(8, 64)
torch.manual_seed(123)
dropout_block(t_in, w_plain).sum().backward()
torch.manual_seed(123)
checkpoint(dropout_block, t_in, w_ckpt).sum().backward()
assert torch.allclose(w_plain.grad, w_ckpt.grad, atol=1e-6), \
    "The recomputation drew a different dropout mask - save and restore the RNG state!"
print("Recomputed dropout mask matches the forward pass")
print("PASSED\n")

# 4. Training works correctly
print("=== Training Test ===")
model_train = DeepNetwork(dim=64, num_blocks=24, use_checkpoint=True)
optimizer = torch.optim.Adam(model_train.parameters(), lr=1e-3)
X = torch.randn(32, 64)
y = torch.randn(32, 1)

# X is plain data (requires_grad=False), exactly like a batch from a DataLoader.
# Every checkpointed block must still receive gradients, not just the head.
F.mse_loss(model_train(X), y).backward()
no_grad = [n for n, p in model_train.named_parameters() if p.grad is None or not p.grad.abs().sum()]
assert not no_grad, f"{len(no_grad)} parameters got no gradient, e.g. {no_grad[:3]}"
print("Every parameter receives a gradient with plain (no-grad) input")

losses = []
for step in range(30):
    optimizer.zero_grad()
    pred = model_train(X)
    loss = F.mse_loss(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease with checkpointed training!"
print("PASSED\n")

print("All tests passed!")


In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gradient-checkpointing", CheckpointFunction=CheckpointFunction, checkpoint=checkpoint)